<a href="https://colab.research.google.com/github/AymaneAmri-MD/drug-design-morphine-paclitaxel/blob/main/notebooks/02_docking_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Docking : re-docking de validation, puis morphine et atropine
Notebook autonome pour **Google Colab** (Exécution → Tout exécuter). Durée attendue : quelques minutes (ligands petits et peu flexibles).

| Cible | PDB | Ligand co-cristallisé (re-docking) | Molécule à docker |
|---|---|---|---|
| Récepteur opioïde µ (chaîne A) | **5C1M** | BU72 (Agoniste) | Morphine |
| Récepteur muscarinique M2 (chaîne A) | **3UON** | QNB (Antagoniste) | Atropine (L-hyoscyamine) |

**Principe :** Pour chaque cible, on isole le ligand co-cristallisé, on le **re-docke** avec Vina et on mesure le RMSD par rapport à sa pose expérimentale. Si le RMSD de la meilleure pose est < 2 Å, le protocole est validé pour cette cible. Seulement ensuite, on interprète le docking de la morphine et de l'atropine.

Un **agoniste** se fixe sur un récepteur et l'active (ex. la morphine sur 5C1M). Un **antagoniste** se fixe sur le site sans l'activer et bloque l'action de l'agoniste naturel (ex. l'atropine sur 3UON). Le ligand co-cristallisé de chaque structure sert à valider le protocole avant de docker nos molécules.

**Cellule 1 · Installation.** Installe les outils : RDKit (chimie), Vina (docking), Meeko (préparation des ligands), Biopython (lecture des PDB) et Open Babel (préparation du récepteur). Les outils de compilation requis par Vina sont installés d'abord. À lancer une seule fois par session, sa durée est de plusieurs minutes.

In [1]:
# 1. Installation (Colab)
import sys
if 'google.colab' in sys.modules:
    !apt-get -qq install -y openbabel swig libboost-all-dev > /dev/null
    !pip install -q rdkit meeko gemmi scipy biopython requests
    !pip install -q vina

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


**Cellule 2 · Imports et configuration.** Charge les bibliothèques, fixe le seed (42) pour la reproductibilité, et définit les deux cibles (PDB, chaîne, exhaustiveness) et les deux molécules à docker. C'est la seule cellule à modifier pour changer de cible ou de molécule.

In [2]:
# 2. Imports et configuration
import shutil, subprocess, time
from pathlib import Path
import numpy as np, pandas as pd, requests
from Bio.PDB import PDBParser, PDBIO, Select
from rdkit import Chem, rdBase
from rdkit.Chem import AllChem, rdMolAlign
from meeko import MoleculePreparation, PDBQTWriterLegacy, PDBQTMolecule, RDKitMolCreate
from vina import Vina

SEED = 42
WORK = Path('results_02'); WORK.mkdir(exist_ok=True)

TARGETS = {
    'mu_opioid':     {'pdb': '5C1M', 'chain': 'A', 'ligand_resname': None, 'exhaustiveness': 16},
    'muscarinic_m2': {'pdb': '3UON', 'chain': 'A', 'ligand_resname': None, 'exhaustiveness': 16},
}
TO_DOCK = {  # nom : (CID PubChem ou nom de composé, cible)
    'morphine': (5288826,      'mu_opioid'),
    'atropine': ('hyoscyamine', 'muscarinic_m2'),  # L-hyoscyamine = énantiomère actif de l'atropine
}
# Groupes HETATM ignorés lors de la détection automatique du ligand
EXCLUDE = {'HOH','NA','CL','K','MG','ZN','CA','MN','SO4','PO4','GOL','EDO','PEG','ACT','DMS','MES',
           'GDP','GTP','GNP','GSP','ANP','ATP','ADP','NAG','BMA','MAN','FUC'}
print('RDKit', rdBase.rdkitVersion, '| seed', SEED)

RDKit 2026.09.1 | seed 42


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Cellule 3a · Fonctions « protéines ».** Définit les fonctions qui téléchargent un PDB depuis la RCSB, détectent le ligand co-cristallisé, séparent protéine et ligand, et préparent le récepteur au format PDBQT. Elle ne fait que définir des fonctions : aucun affichage.

In [3]:
# 3a. Fonctions : structures protéiques
def download_pdb(pdb_id):
    path = WORK / f'{pdb_id}.pdb'
    if not path.exists():
        r = requests.get(f'https://files.rcsb.org/download/{pdb_id}.pdb', timeout=60)
        r.raise_for_status()
        path.write_text(r.text)
    return path

def list_hetero(structure, chain_id):
    out = []
    for res in structure[0][chain_id]:
        name = res.get_resname().strip()
        if res.id[0].startswith('H_') and name not in EXCLUDE:
            n_heavy = sum(1 for a in res if (a.element or '').strip() != 'H')
            if n_heavy >= 8:
                out.append((name, res.id, n_heavy))
    return sorted(out, key=lambda x: -x[2])

class ProteinSelect(Select):
    def __init__(self, chain): self.chain = chain
    def accept_chain(self, chain): return chain.id == self.chain
    def accept_residue(self, res): return res.id[0] == ' '
    def accept_atom(self, atom): return atom.get_altloc() in (' ', 'A')

class LigandSelect(Select):
    def __init__(self, chain, rid): self.chain, self.rid = chain, rid
    def accept_chain(self, chain): return chain.id == self.chain
    def accept_residue(self, res): return res.id == self.rid
    def accept_atom(self, atom): return atom.get_altloc() in (' ', 'A')

def split_complex(pdb_path, chain_id, resname=None):
    s = PDBParser(QUIET=True).get_structure('x', str(pdb_path))
    if chain_id not in [c.id for c in s[0]]:
        raise RuntimeError(f"Chaîne {chain_id} absente de {pdb_path.name} : chaînes = {[c.id for c in s[0]]}")
    hets = list_hetero(s, chain_id)
    print('  Ligands candidats dans la chaîne', chain_id, '(nom, id, atomes lourds) :', hets[:6])
    if not hets:
        raise RuntimeError('Aucun ligand trouvé dans cette chaîne : change la chaîne dans TARGETS.')
    name, rid, n = next(h for h in hets if resname is None or h[0] == resname)
    io = PDBIO(); io.set_structure(s)
    stem = pdb_path.stem
    prot = WORK / f'{stem}_{chain_id}_protein.pdb'
    lig = WORK / f'{stem}_{name}_crystal.pdb'
    io.save(str(prot), ProteinSelect(chain_id))
    io.save(str(lig), LigandSelect(chain_id, rid))
    return prot, lig, name

def prepare_receptor(prot_pdb):
    out = prot_pdb.with_suffix('.pdbqt')
    cmd = ['obabel', str(prot_pdb), '-O', str(out), '-xr', '-p', '7.4']
    res = subprocess.run(cmd, capture_output=True, text=True)
    if not out.exists() or out.stat().st_size == 0:
        raise RuntimeError('Échec de la préparation du récepteur : ' + res.stderr[-500:])
    return out

**Cellule 3b · Fonctions « ligands et docking ».** Définit les fonctions qui reconstruisent le ligand co-cristallisé avec ses liaisons, calculent la boîte de docking, convertissent en PDBQT, lancent Vina et calculent le RMSD. Aucun affichage non plus.

In [4]:
# 3b. Fonctions : ligands, docking, RMSD
def crystal_ligand_mol(lig_pdb, resname):
    """Ligand expérimental avec ordres de liaison corrects (modèle : SDF idéal de la RCSB)."""
    raw = Chem.MolFromPDBFile(str(lig_pdb), sanitize=False, removeHs=False, proximityBonding=True)
    r = requests.get(f'https://files.rcsb.org/ligands/download/{resname}_ideal.sdf', timeout=60)
    r.raise_for_status()
    template = Chem.MolFromMolBlock(r.text, removeHs=True)
    mol = AllChem.AssignBondOrdersFromTemplate(template, raw)
    Chem.SanitizeMol(mol)
    return mol  # atomes lourds, coordonnées cristallographiques

def pocket_box(mol, padding=12.0, minimum=22.0):
    xyz = mol.GetConformer().GetPositions()
    center = xyz.mean(axis=0)
    size = np.maximum(xyz.max(axis=0) - xyz.min(axis=0) + padding, minimum)
    return center, size

def to_pdbqt(mol_h):
    setups = MoleculePreparation().prepare(mol_h)
    pdbqt, ok, err = PDBQTWriterLegacy.write_string(setups[0])
    if not ok:
        raise RuntimeError(err)
    return pdbqt

def run_vina(rec_pdbqt, lig_pdbqt, center, box, exhaustiveness, n_poses=10):
    t0 = time.time()
    box = np.minimum(box, 30)
    n_rec = sum(1 for l in open(rec_pdbqt) if l.startswith(('ATOM', 'HETATM')))
    print(f'  récepteur : {n_rec} atomes | centre {np.round(center, 1)} | boîte {np.round(box, 1)} Å')
    v = Vina(sf_name='vina', cpu=0, seed=SEED, verbosity=1)
    v.set_receptor(str(rec_pdbqt))
    v.set_ligand_from_string(lig_pdbqt)
    v.compute_vina_maps(center=[float(x) for x in center], box_size=[float(x) for x in box])
    v.dock(exhaustiveness=exhaustiveness, n_poses=n_poses)
    print(f'  terminé en {time.time()-t0:.0f}s | scores : {np.round(v.energies(n_poses=n_poses)[:, 0], 2)}')
    return v

def pose_rmsds(poses_pdbqt, ref_heavy):
    pm = PDBQTMolecule(poses_pdbqt, is_dlg=False, skip_typing=True)
    docked = Chem.RemoveAllHs(RDKitMolCreate.from_pdbqt_mol(pm)[0])
    ref_id = ref_heavy.GetConformer().GetId()
    return [rdMolAlign.CalcRMS(docked, ref_heavy, prbId=c.GetId(), refId=ref_id)
            for c in docked.GetConformers()]

## 3. Préparation des cibles
Télécharge chaque structure, isole la chaîne protéique, extrait le ligand co-cristallisé et définit la boîte de docking autour de lui. **Lis les ligands détectés** : le ligand choisi doit être l'agoniste ou l'antagoniste attendu (BU72 pour 5C1M, QNB pour 3UON), pas un lipide ou un cofacteur.

**Ce que fait la cellule :** Pour chaque cible, télécharge le PDB, sépare la protéine et le ligand co-cristallisé, prépare le récepteur et calcule la boîte de docking. Elle affiche les ligands candidats, celui retenu, le centre et la taille de la boîte.

In [5]:
prepared = {}
for key, cfg in TARGETS.items():
    print(f"=== {key} : {cfg['pdb']} chaîne {cfg['chain']}")
    pdb = download_pdb(cfg['pdb'])
    prot, lig, resname = split_complex(pdb, cfg['chain'], cfg['ligand_resname'])
    rec = prepare_receptor(prot)
    crystal = crystal_ligand_mol(lig, resname)
    center, box = pocket_box(crystal)
    prepared[key] = dict(receptor=rec, crystal=crystal, resname=resname, center=center, box=box)
    print(f"  Ligand retenu : {resname} ({crystal.GetNumAtoms()} atomes lourds)")
    print(f"  Centre : {np.round(center, 1)} | boîte : {np.round(box, 1)} Å")

=== mu_opioid : 5C1M chaîne A


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  Ligands candidats dans la chaîne A (nom, id, atomes lourds) : [('VF1', ('H_VF1', 407, ' '), 32), ('CLR', ('H_CLR', 403, ' '), 28), ('P6G', ('H_P6G', 405, ' '), 19), ('OLC', ('H_OLC', 402, ' '), 18), ('OLC', ('H_OLC', 401, ' '), 16), ('P6G', ('H_P6G', 406, ' '), 13)]


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
[RDKit] WARNING:[22:02:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[RDKit] WARNING:[22:02:51] WARNING: More than one matching pattern found - picking one
[RDKit] WARNING:


  Ligand retenu : VF1 (32 atomes lourds)
  Centre : [  1.3  16.4 -59.1] | boîte : [22. 22. 22.] Å
=== muscarinic_m2 : 3UON chaîne A
  Ligands candidats dans la chaîne A (nom, id, atomes lourds) : [('QNB', ('H_QNB', 1162, ' '), 25), ('BGC', ('H_BGC', 1163, ' '), 12)]


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
[RDKit] WARNING:[22:02:52] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[RDKit] WARNING:[22:02:52] WARNING: More than one matching pattern found - picking one
[RDKit] WARNING:


  Ligand retenu : QNB (25 atomes lourds)
  Centre : [ 7.8  0.3 -4. ] | boîte : [22. 22. 22.] Å


## 4. Re-docking (validation)
Le ligand co-cristallisé est redocké dans sa propre poche. Critère : **RMSD de la meilleure pose < 2 Å**. Si une cible échoue (RMSD ≥ 2 Å), c'est un résultat à rapporter honnêtement, pas à cacher.

**Ce que fait la cellule :** redocke le ligand co-cristallisé de chaque cible (10 poses) et le compare à sa position expérimentale. Le tableau donne le score de la pose 1, son RMSD, le RMSD minimal sur les 10 poses et la validation (< 2 Å).

In [6]:
redock = []
for key, p in prepared.items():
    cfg = TARGETS[key]
    mol_h = AllChem.AddHs(p['crystal'], addCoords=True)
    v = run_vina(p['receptor'], to_pdbqt(mol_h), p['center'], p['box'], cfg['exhaustiveness'])
    energies = v.energies(n_poses=10)[:, 0]
    rms = pose_rmsds(v.poses(n_poses=10), p['crystal'])
    redock.append({'cible': key, 'ligand': p['resname'], 'score_pose1': round(energies[0], 2),
                   'RMSD_pose1': round(rms[0], 2), 'RMSD_min_10poses': round(min(rms), 2),
                   'validé (<2 Å)': min(rms) < 2.0})
redock_df = pd.DataFrame(redock)
redock_df

  récepteur : 2820 atomes | centre [  1.3  16.4 -59.1] | boîte [22. 22. 22.] Å
  terminé en 117s | scores : [-12.18]
  récepteur : 4180 atomes | centre [ 7.8  0.3 -4. ] | boîte [22. 22. 22.] Å
  terminé en 106s | scores : [-11.47 -10.35  -9.87  -8.8 ]


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,cible,ligand,score_pose1,RMSD_pose1,RMSD_min_10poses,validé (<2 Å)
0,mu_opioid,VF1,-12.18,0.57,0.57,True
1,muscarinic_m2,QNB,-11.47,0.63,0.63,True


## 5. Docking de la morphine et de l'atropine
Les SMILES isomériques viennent de PubChem (CID pour la morphine, nom « hyoscyamine » pour l'atropine : on dock l'énantiomère actif L-hyoscyamine, alors que l'atropine est le mélange racémique). Conformère de départ : RDKit + MMFF, seed fixé. Si le re-docking d'une cible n'est pas validé, le résultat correspondant est à interpréter avec une très grande prudence.

**Ce que fait la cellule :** récupère les SMILES sur PubChem, génère un conformère 3D de chaque molécule, la docke dans la boîte de sa cible et enregistre les 10 poses (fichiers .pdbqt). Le tableau final donne le meilleur score et les trois meilleurs.

In [7]:
def pubchem_smiles(ident):
    """SMILES isomérique depuis PubChem, par CID (int) ou par nom (str)."""
    kind = 'cid' if isinstance(ident, int) else 'name'
    url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/{kind}/{ident}/property/IsomericSMILES/TXT'
    r = requests.get(url, timeout=30); r.raise_for_status()
    return r.text.strip().splitlines()[0]

results = []
for name, (cid, target) in TO_DOCK.items():
    smi = pubchem_smiles(cid)
    print(name, '->', smi)
    assert '@' in smi, f'{name} : stéréochimie absente'
    mol_h = Chem.AddHs(Chem.MolFromSmiles(smi))
    AllChem.EmbedMolecule(mol_h, randomSeed=SEED)
    AllChem.MMFFOptimizeMolecule(mol_h, maxIters=2000)
    p, cfg = prepared[target], TARGETS[target]
    v = run_vina(p['receptor'], to_pdbqt(mol_h), p['center'], p['box'], cfg['exhaustiveness'])
    energies = v.energies(n_poses=10)[:, 0]
    v.write_poses(str(WORK / f'{name}_{cfg["pdb"]}_poses.pdbqt'), n_poses=10, overwrite=True)
    results.append({'ligand': name, 'cible': f"{cfg['pdb']} ch.{cfg['chain']}",
                    'meilleur_score (kcal/mol)': round(energies[0], 2),
                    'scores_top3': [round(e, 2) for e in energies[:3]]})
results_df = pd.DataFrame(results)
results_df

morphine -> CN1CC[C@]23[C@@H]4[C@H]1CC5=C2C(=C(C=C5)O)O[C@H]3[C@H](C=C4)O
  récepteur : 2820 atomes | centre [  1.3  16.4 -59.1] | boîte [22. 22. 22.] Å
  terminé en 29s | scores : [-8.29 -8.26 -8.01 -7.68 -7.5  -7.47 -7.37 -7.32 -7.27 -7.22]
atropine -> CN1[C@@H]2CC[C@H]1CC(C2)OC(=O)[C@H](CO)C3=CC=CC=C3
  récepteur : 4180 atomes | centre [ 7.8  0.3 -4. ] | boîte [22. 22. 22.] Å
  terminé en 75s | scores : [-9.4  -9.02 -8.86 -7.51 -7.43 -7.02 -6.88 -6.77]


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,ligand,cible,meilleur_score (kcal/mol),scores_top3
0,morphine,5C1M ch.A,-8.29,"[-8.29, -8.26, -8.01]"
1,atropine,3UON ch.A,-9.40,"[-9.4, -9.02, -8.86]"


## 6. Sauvegarde
Poses (PDBQT) et tableaux dans `results_02/`. Télécharge l'archive, puis place-la dans le dépôt GitHub (dossier `results/`) si les fichiers sont légers.

**Ce que fait la cellule :** enregistre les deux tableaux en CSV, crée une archive `results_02.zip` et la télécharge sur ton ordinateur (Colab uniquement).

In [8]:
redock_df.to_csv(WORK / 'redocking.csv', index=False)
results_df.to_csv(WORK / 'docking_scores.csv', index=False)
archive = shutil.make_archive('results_02', 'zip', WORK)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Limites
- Les scores Vina sont des estimations grossières de l'énergie de liaison : à ne pas comparer entre cibles différentes ni à des affinités mesurées.
- Le récepteur est rigide et préparé automatiquement (hydrogènes ajoutés à pH 7,4 par Open Babel, sans optimisation des états de protonation des résidus).
- Morphine et atropine sont dockées sous forme neutre ; leur amine tertiaire est en réalité surtout protonée à pH physiologique et forme un pont salin avec un aspartate du récepteur (important pour la pose).
- 5C1M est un récepteur µ lié à un agoniste (état actif, nanobody exclu) ; 3UON est un récepteur M2 lié à un antagoniste (état inactif, fusion avec le lysozyme T4). Docker un ligand d'une autre classe fonctionnelle que le ligand cristallisé reste une approximation.
- On dock l'énantiomère L-hyoscyamine ; l'atropine du commerce est un mélange racémique.
- Seed fixé à 42, mais les résultats peuvent varier légèrement selon les versions de Vina et de Meeko.

## Étape suivante
`03_generative.ipynb` : génération d'analogues puis filtrage par docking.